سلول ۱ — مسیرها و فایل‌های ورودی

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

PROJECT_ROOT = Path(".")

PAIR_DIR = PROJECT_ROOT / "Data_proc" / "pairs"
RAW_FASTA_DIR = PROJECT_ROOT / "Data_raw" / "uniprot" / "uniprot_fasta"
INTERIM_UNIPROT_DIR = PROJECT_ROOT / "Data_interim" / "uniprot"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

for d in [INTERIM_UNIPROT_DIR, QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PAIRS_ALL_PATH = PAIR_DIR / "pairs_all_final.csv"
REQ_ACCESSIONS_PATH = PAIR_DIR / "required_accessions_final.csv"
REQ_ENZYMES_PATH = PAIR_DIR / "required_enzymes_final.csv"
REQ_SUBSTRATES_PATH = PAIR_DIR / "required_substrates_final.csv"

print("pairs_all exists:", PAIRS_ALL_PATH.exists(), PAIRS_ALL_PATH)
print("required_accessions exists:", REQ_ACCESSIONS_PATH.exists(), REQ_ACCESSIONS_PATH)
print("required_enzymes exists:", REQ_ENZYMES_PATH.exists(), REQ_ENZYMES_PATH)
print("required_substrates exists:", REQ_SUBSTRATES_PATH.exists(), REQ_SUBSTRATES_PATH)
print("raw fasta dir exists:", RAW_FASTA_DIR.exists(), RAW_FASTA_DIR)

pairs_all = pd.read_csv(PAIRS_ALL_PATH, dtype=str, low_memory=False)
required_accessions = pd.read_csv(REQ_ACCESSIONS_PATH, dtype=str, low_memory=False)
required_enzymes = pd.read_csv(REQ_ENZYMES_PATH, dtype=str, low_memory=False)
required_substrates = pd.read_csv(REQ_SUBSTRATES_PATH, dtype=str, low_memory=False)

print("pairs_all:", pairs_all.shape)
print("required_accessions:", required_accessions.shape)
print("required_enzymes:", required_enzymes.shape)
print("required_substrates:", required_substrates.shape)

display(pairs_all.head())
display(required_accessions.head())

سلول ۲ — توابع خواندن و اعتبارسنجی FASTA

In [ ]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWYBXZUOJ*-")

def normalize_ac(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return np.nan
    x = re.split(r"[;,|\s]", x)[0].strip()
    x = re.sub(r"-\d+$", "", x)
    return x.upper()


def extract_ac_from_header(header: str):
    """
    UniProt FASTA header usually:
    >sp|P12345|NAME_HUMAN ...
    """
    if not isinstance(header, str):
        return np.nan
    
    h = header.strip()
    if h.startswith(">"):
        h = h[1:].strip()
    
    if not h:
        return np.nan
    
    if "|" in h:
        parts = h.split("|")
        if len(parts) >= 2:
            return normalize_ac(parts[1])
    
    return normalize_ac(h.split()[0])


def read_single_fasta(path: Path):
    """
    Read one fasta file.
    Returns dict with status.
    """
    try:
        text = path.read_text(errors="ignore")
    except Exception as e:
        return {
            "ok": False,
            "error": f"read_error: {e}",
            "header": "",
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    if not text.strip():
        return {
            "ok": False,
            "error": "empty_file",
            "header": "",
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    
    if not lines:
        return {
            "ok": False,
            "error": "no_nonempty_lines",
            "header": "",
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    if not lines[0].startswith(">"):
        return {
            "ok": False,
            "error": "missing_header",
            "header": lines[0][:100],
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    header = lines[0]
    seq = "".join(lines[1:]).replace(" ", "").replace("\t", "").upper()
    
    if not seq:
        return {
            "ok": False,
            "error": "empty_sequence",
            "header": header,
            "seq": "",
            "header_ac": extract_ac_from_header(header),
            "length": 0,
        }
    
    invalid_chars = sorted(set(seq) - VALID_AA)
    
    if invalid_chars:
        return {
            "ok": False,
            "error": "invalid_amino_acids:" + "".join(invalid_chars),
            "header": header,
            "seq": seq,
            "header_ac": extract_ac_from_header(header),
            "length": len(seq),
        }
    
    return {
        "ok": True,
        "error": "",
        "header": header,
        "seq": seq,
        "header_ac": extract_ac_from_header(header),
        "length": len(seq),
    }

سلول ۳ — ساخت index از FASTAهای موجود

In [ ]:
# Detect all single FASTA files
fasta_files = sorted(
    list(RAW_FASTA_DIR.glob("*.fasta")) +
    list(RAW_FASTA_DIR.glob("*.fa")) +
    list(RAW_FASTA_DIR.glob("*.faa"))
)

print("FASTA files found:", len(fasta_files))

index_rows = []

for i, path in enumerate(fasta_files):
    if i % 1000 == 0:
        print("processing", i, "/", len(fasta_files))
    
    file_stem_ac = normalize_ac(path.stem)
    rec = read_single_fasta(path)
    
    index_rows.append({
        "file_path": str(path),
        "file_name": path.name,
        "file_stem_ac": file_stem_ac,
        "header_ac": rec["header_ac"],
        "ok": rec["ok"],
        "error": rec["error"],
        "length": rec["length"],
        "header": rec["header"],
    })

fasta_index = pd.DataFrame(index_rows)

print("FASTA index:", fasta_index.shape)
print("OK FASTA:", int(fasta_index["ok"].sum()) if len(fasta_index) else 0)
print("Bad FASTA:", int((~fasta_index["ok"]).sum()) if len(fasta_index) else 0)

display(fasta_index.head())
display(fasta_index["error"].value_counts(dropna=False).head(20))

سلول ۴ — اتصال accessionهای موردنیاز به FASTA index

In [ ]:
# Normalize required accessions
if "accession" in required_accessions.columns:
    req_col = "accession"
elif "uniprot_ac" in required_accessions.columns:
    req_col = "uniprot_ac"
else:
    req_col = required_accessions.columns[0]

required_accessions["accession"] = required_accessions[req_col].map(normalize_ac)
required_accessions = (
    required_accessions[["accession"]]
    .dropna()
    .drop_duplicates()
    .sort_values("accession")
    .reset_index(drop=True)
)

# Prefer file_stem_ac matching; header_ac is secondary evidence
ok_index = fasta_index[fasta_index["ok"]].copy()

# If duplicate FASTA files for same AC exist, keep first
ok_by_stem = (
    ok_index
    .dropna(subset=["file_stem_ac"])
    .drop_duplicates(subset=["file_stem_ac"], keep="first")
    .set_index("file_stem_ac")
)

coverage_rows = []

for acc in required_accessions["accession"]:
    if acc in ok_by_stem.index:
        row = ok_by_stem.loc[acc]
        coverage_rows.append({
            "accession": acc,
            "has_fasta": True,
            "fasta_ok": True,
            "file_path": row["file_path"],
            "file_name": row["file_name"],
            "length": int(row["length"]),
            "header_ac": row["header_ac"],
            "error": "",
        })
    else:
        # Check if present but bad
        bad_match = fasta_index[fasta_index["file_stem_ac"] == acc]
        if len(bad_match):
            err = ";".join(bad_match["error"].astype(str).unique())
            coverage_rows.append({
                "accession": acc,
                "has_fasta": True,
                "fasta_ok": False,
                "file_path": bad_match.iloc[0]["file_path"],
                "file_name": bad_match.iloc[0]["file_name"],
                "length": int(bad_match.iloc[0]["length"]),
                "header_ac": bad_match.iloc[0]["header_ac"],
                "error": err,
            })
        else:
            coverage_rows.append({
                "accession": acc,
                "has_fasta": False,
                "fasta_ok": False,
                "file_path": "",
                "file_name": "",
                "length": 0,
                "header_ac": "",
                "error": "missing_file",
            })

fasta_coverage = pd.DataFrame(coverage_rows)

missing_required = fasta_coverage[~fasta_coverage["has_fasta"]].copy()
bad_required = fasta_coverage[
    fasta_coverage["has_fasta"] & (~fasta_coverage["fasta_ok"])
].copy()

print("Required accessions:", len(fasta_coverage))
print("Covered OK:", int(fasta_coverage["fasta_ok"].sum()))
print("Missing:", len(missing_required))
print("Bad:", len(bad_required))

display(fasta_coverage.head())
display(missing_required.head())
display(bad_required.head())

سلول ۵ — QC بر اساس نقش enzyme/substrate

In [ ]:
def prepare_role_df(df, col_guess):
    out = df.copy()
    if col_guess in out.columns:
        col = col_guess
    elif "accession" in out.columns:
        col = "accession"
    elif "uniprot_ac" in out.columns:
        col = "uniprot_ac"
    else:
        col = out.columns[0]
    
    out["accession"] = out[col].map(normalize_ac)
    out = out[["accession"]].dropna().drop_duplicates()
    return out


enz_req = prepare_role_df(required_enzymes, "enz_ac")
sub_req = prepare_role_df(required_substrates, "sub_ac")

enz_cov = enz_req.merge(fasta_coverage, on="accession", how="left")
sub_cov = sub_req.merge(fasta_coverage, on="accession", how="left")

fasta_coverage_qc = pd.DataFrame([
    {
        "dataset": "all_required_accessions",
        "n_required": len(fasta_coverage),
        "n_fasta_ok": int(fasta_coverage["fasta_ok"].sum()),
        "n_missing": int((~fasta_coverage["has_fasta"]).sum()),
        "n_bad": int((fasta_coverage["has_fasta"] & ~fasta_coverage["fasta_ok"]).sum()),
        "coverage_fraction": float(fasta_coverage["fasta_ok"].mean()) if len(fasta_coverage) else np.nan,
        "mean_length_ok": float(fasta_coverage.loc[fasta_coverage["fasta_ok"], "length"].mean()) if fasta_coverage["fasta_ok"].any() else np.nan,
        "median_length_ok": float(fasta_coverage.loc[fasta_coverage["fasta_ok"], "length"].median()) if fasta_coverage["fasta_ok"].any() else np.nan,
    },
    {
        "dataset": "required_enzymes",
        "n_required": len(enz_cov),
        "n_fasta_ok": int(enz_cov["fasta_ok"].fillna(False).sum()),
        "n_missing": int((enz_cov["has_fasta"].fillna(False) == False).sum()),
        "n_bad": int((enz_cov["has_fasta"].fillna(False) & (enz_cov["fasta_ok"].fillna(False) == False)).sum()),
        "coverage_fraction": float(enz_cov["fasta_ok"].fillna(False).mean()) if len(enz_cov) else np.nan,
        "mean_length_ok": float(enz_cov.loc[enz_cov["fasta_ok"] == True, "length"].mean()) if (enz_cov["fasta_ok"] == True).any() else np.nan,
        "median_length_ok": float(enz_cov.loc[enz_cov["fasta_ok"] == True, "length"].median()) if (enz_cov["fasta_ok"] == True).any() else np.nan,
    },
    {
        "dataset": "required_substrates",
        "n_required": len(sub_cov),
        "n_fasta_ok": int(sub_cov["fasta_ok"].fillna(False).sum()),
        "n_missing": int((sub_cov["has_fasta"].fillna(False) == False).sum()),
        "n_bad": int((sub_cov["has_fasta"].fillna(False) & (sub_cov["fasta_ok"].fillna(False) == False)).sum()),
        "coverage_fraction": float(sub_cov["fasta_ok"].fillna(False).mean()) if len(sub_cov) else np.nan,
        "mean_length_ok": float(sub_cov.loc[sub_cov["fasta_ok"] == True, "length"].mean()) if (sub_cov["fasta_ok"] == True).any() else np.nan,
        "median_length_ok": float(sub_cov.loc[sub_cov["fasta_ok"] == True, "length"].median()) if (sub_cov["fasta_ok"] == True).any() else np.nan,
    },
])

display(fasta_coverage_qc)

این خروجی یعنی FASTA coverage فعلاً کامل نیست و نباید هنوز برویم سراغ ساخت embedding نهایی

قدم بعدی: چک missing و bad accessionها

In [ ]:
print("Missing required FASTA:", len(missing_required))
display(missing_required.head(50))

print("Bad required FASTA:", len(bad_required))
display(bad_required)

print("Missing enzymes:")
missing_enz = enz_cov[enz_cov["fasta_ok"].fillna(False) == False]
display(missing_enz.head(50))

print("Missing substrates:")
missing_sub = sub_cov[sub_cov["fasta_ok"].fillna(False) == False]
display(missing_sub.head(50))

In [ ]:
all_fasta_recursive = sorted(
    list(RAW_FASTA_DIR.glob("**/*.fasta")) +
    list(RAW_FASTA_DIR.glob("**/*.fa")) +
    list(RAW_FASTA_DIR.glob("**/*.faa"))
)

print("Top-level FASTA files:", len(fasta_files))
print("Recursive FASTA files:", len(all_fasta_recursive))

for p in all_fasta_recursive[:20]:
    print(p.relative_to(PROJECT_ROOT))

قدم ۱ — پیدا کردن accessionهای ترکیبی یا مشکوک

In [ ]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(".")

PAIR_DIR = PROJECT_ROOT / "Data_proc" / "pairs"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

pairs_all_path = PAIR_DIR / "pairs_all_final.csv"
pairs_all = pd.read_csv(pairs_all_path, dtype=str, low_memory=False)

def is_suspicious_ac(x):
    if pd.isna(x):
        return True
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return True
    # suspicious separators or whitespace
    if re.search(r"[#;,|\s]", x):
        return True
    return False

suspicious_rows = pairs_all[
    pairs_all["enz_ac"].map(is_suspicious_ac) |
    pairs_all["sub_ac"].map(is_suspicious_ac)
].copy()

print("Suspicious rows:", suspicious_rows.shape)
display(suspicious_rows[[
    "pair_id", "group_id", "enzyme_class",
    "enz_ac", "sub_ac", "enz_gene", "sub_gene",
    "label", "source", "pmid"
]].head(100))

suspicious_rows.to_csv(
    QC_DIR / "suspicious_accessions_in_pairs_all_final.csv",
    index=False
)

قدم ۲ — اصلاح normalize_ac برای جداکننده #

In [ ]:
def normalize_ac_strict(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return np.nan
    
    # Split compound/multiple identifiers
    x = re.split(r"[#;,|\s]", x)[0].strip()
    
    # Remove UniProt isoform suffix
    x = re.sub(r"-\d+$", "", x)
    
    return x.upper()

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

PROJECT_ROOT = Path(".")

PAIR_DIR = PROJECT_ROOT / "Data_proc" / "pairs"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

pairs_path = PAIR_DIR / "pairs_all_final.csv"

pairs = pd.read_csv(pairs_path, dtype=str, low_memory=False)

# Backup
pairs.to_csv(PAIR_DIR / "pairs_all_final_before_ac_strict_fix.csv", index=False)

def normalize_ac_strict(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return np.nan
    x = re.split(r"[#;,|\s]", x)[0].strip()
    x = re.sub(r"-\d+$", "", x)
    return x.upper()

pairs_fixed = pairs.copy()

pairs_fixed["enz_ac_raw"] = pairs_fixed["enz_ac"]
pairs_fixed["sub_ac_raw"] = pairs_fixed["sub_ac"]

pairs_fixed["enz_ac"] = pairs_fixed["enz_ac"].map(normalize_ac_strict)
pairs_fixed["sub_ac"] = pairs_fixed["sub_ac"].map(normalize_ac_strict)

pairs_fixed["enzyme_class"] = pairs_fixed["enzyme_class"].astype(str).str.strip()

pairs_fixed["pair_id"] = (
    pairs_fixed["enzyme_class"].astype(str)
    + "|"
    + pairs_fixed["enz_ac"].astype(str)
    + "|"
    + pairs_fixed["sub_ac"].astype(str)
)

pairs_fixed["group_id"] = (
    pairs_fixed["enzyme_class"].astype(str)
    + "|"
    + pairs_fixed["enz_ac"].astype(str)
)

pairs_fixed["label"] = pairs_fixed["label"].astype(int)

# QC after fix
dup_pair_ids = pairs_fixed[
    pairs_fixed.duplicated("pair_id", keep=False)
].sort_values("pair_id")

label_conflict_keys = (
    pairs_fixed.groupby("pair_id")["label"]
    .nunique()
    .reset_index(name="n_labels")
)
label_conflict_keys = label_conflict_keys[label_conflict_keys["n_labels"] > 1]

label_conflicts = pairs_fixed[
    pairs_fixed["pair_id"].isin(label_conflict_keys["pair_id"])
].sort_values("pair_id")

changed_ac_rows = pairs_fixed[
    (pairs_fixed["enz_ac_raw"].astype(str) != pairs_fixed["enz_ac"].astype(str)) |
    (pairs_fixed["sub_ac_raw"].astype(str) != pairs_fixed["sub_ac"].astype(str))
].copy()

print("Pairs fixed shape:", pairs_fixed.shape)
print("Unique pair_id:", pairs_fixed["pair_id"].nunique())
print("Duplicate pair_id rows:", len(dup_pair_ids))
print("Label conflict pair_ids:", len(label_conflict_keys))
print("Rows with changed accession:", len(changed_ac_rows))

display(changed_ac_rows[[
    "pair_id", "enzyme_class",
    "enz_ac_raw", "enz_ac",
    "sub_ac_raw", "sub_ac",
    "enz_gene", "sub_gene",
    "label"
]].head(100))

display(dup_pair_ids.head(50))
display(label_conflicts.head(50))

سلول ۱ — restore اگر فایل اشتباهی ذخیره شده باشد

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

PROJECT_ROOT = Path(".")

PAIR_DIR = PROJECT_ROOT / "Data_proc" / "pairs"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

pairs_path = PAIR_DIR / "pairs_all_final.csv"
backup_path = PAIR_DIR / "pairs_all_final_before_ac_strict_fix.csv"

if backup_path.exists():
    pairs_original = pd.read_csv(backup_path, dtype=str, low_memory=False)
    print("Loaded backup:", backup_path)
else:
    pairs_original = pd.read_csv(pairs_path, dtype=str, low_memory=False)
    print("Loaded current pairs_all_final:", pairs_path)

pairs_original["label"] = pairs_original["label"].astype(int)

print("pairs_original:", pairs_original.shape)
print("unique pair_id:", pairs_original["pair_id"].nunique())
print("duplicates:", pairs_original.duplicated("pair_id").sum())
print("label counts:")
print(pairs_original["label"].value_counts())

سلول ۲ — شناسایی ردیف‌های غیرقابل استفاده برای sequence model

In [ ]:
def has_compound_or_suspicious_ac(x):
    if pd.isna(x):
        return True
    
    x = str(x).strip()
    
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return True
    
    # compound or multiple identifiers
    if re.search(r"[#;,|\s]", x):
        return True
    
    return False


compound_rows = pairs_original[
    pairs_original["enz_ac"].map(has_compound_or_suspicious_ac) |
    pairs_original["sub_ac"].map(has_compound_or_suspicious_ac)
].copy()

model_ready = pairs_original[
    ~(
        pairs_original["enz_ac"].map(has_compound_or_suspicious_ac) |
        pairs_original["sub_ac"].map(has_compound_or_suspicious_ac)
    )
].copy()

print("Original pairs:", pairs_original.shape)
print("Compound/suspicious rows excluded:", compound_rows.shape)
print("Model-ready pairs:", model_ready.shape)

print("\nExcluded label counts:")
print(compound_rows["label"].value_counts(dropna=False))

print("\nExcluded enzyme_class counts:")
print(compound_rows["enzyme_class"].value_counts(dropna=False))

display(compound_rows[[
    "pair_id", "group_id", "enzyme_class",
    "enz_ac", "sub_ac", "enz_gene", "sub_gene",
    "enzyme_type", "label", "source", "pmid"
]].head(100))

سلول ۳ — بازسازی شناسه‌ها فقط برای ردیف‌های سالم

In [ ]:
def normalize_ac_simple(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return np.nan
    
    # فقط isoform suffix را حذف می‌کنیم، نه # را split
    x = re.sub(r"-\d+$", "", x)
    
    return x.upper()


model_ready["enz_ac"] = model_ready["enz_ac"].map(normalize_ac_simple)
model_ready["sub_ac"] = model_ready["sub_ac"].map(normalize_ac_simple)

model_ready["enzyme_class"] = model_ready["enzyme_class"].astype(str).str.strip()

model_ready["pair_id"] = (
    model_ready["enzyme_class"].astype(str)
    + "|"
    + model_ready["enz_ac"].astype(str)
    + "|"
    + model_ready["sub_ac"].astype(str)
)

model_ready["group_id"] = (
    model_ready["enzyme_class"].astype(str)
    + "|"
    + model_ready["enz_ac"].astype(str)
)

model_ready["label"] = model_ready["label"].astype(int)

dup_model = model_ready[
    model_ready.duplicated("pair_id", keep=False)
].sort_values("pair_id")

label_conflict_keys = (
    model_ready.groupby("pair_id")["label"]
    .nunique()
    .reset_index(name="n_labels")
)

label_conflict_keys = label_conflict_keys[label_conflict_keys["n_labels"] > 1]

label_conflicts_model = model_ready[
    model_ready["pair_id"].isin(label_conflict_keys["pair_id"])
].sort_values("pair_id")

print("Model-ready shape:", model_ready.shape)
print("Unique pair_id:", model_ready["pair_id"].nunique())
print("Duplicate pair_id rows:", len(dup_model))
print("Label conflict pair_ids:", len(label_conflict_keys))

print("\nLabel counts:")
print(model_ready["label"].value_counts())

print("\nEnzyme class counts:")
print(model_ready["enzyme_class"].value_counts())

display(dup_model.head(50))
display(label_conflicts_model.head(50))

سلول ۴ — ذخیره نسخه model-ready و accession list جدید

In [ ]:
if len(dup_model) == 0 and len(label_conflict_keys) == 0:
    
    model_ready.to_csv(
        PAIR_DIR / "pairs_all_model_ready.csv",
        index=False
    )
    
    model_ready[model_ready["label"] == 1].to_csv(
        PAIR_DIR / "pairs_positive_model_ready.csv",
        index=False
    )
    
    model_ready[model_ready["label"] == 0].to_csv(
        PAIR_DIR / "pairs_negative_model_ready.csv",
        index=False
    )
    
    compound_rows.to_csv(
        QC_DIR / "excluded_compound_accession_pairs.csv",
        index=False
    )
    
    required_enzymes_model = (
        model_ready[["enz_ac"]]
        .dropna()
        .drop_duplicates()
        .sort_values("enz_ac")
        .reset_index(drop=True)
    )
    
    required_substrates_model = (
        model_ready[["sub_ac"]]
        .dropna()
        .drop_duplicates()
        .sort_values("sub_ac")
        .reset_index(drop=True)
    )
    
    required_accessions_model = pd.DataFrame({
        "accession": sorted(
            set(required_enzymes_model["enz_ac"].astype(str)) |
            set(required_substrates_model["sub_ac"].astype(str))
        )
    })
    
    required_enzymes_model.to_csv(
        PAIR_DIR / "required_enzymes_model_ready.csv",
        index=False
    )
    
    required_substrates_model.to_csv(
        PAIR_DIR / "required_substrates_model_ready.csv",
        index=False
    )
    
    required_accessions_model.to_csv(
        PAIR_DIR / "required_accessions_model_ready.csv",
        index=False
    )
    
    pd.DataFrame([{
        "n_original_pairs": len(pairs_original),
        "n_excluded_compound_or_suspicious": len(compound_rows),
        "n_model_ready_pairs": len(model_ready),
        "n_model_ready_positive": int((model_ready["label"] == 1).sum()),
        "n_model_ready_negative": int((model_ready["label"] == 0).sum()),
        "n_model_ready_E3": int((model_ready["enzyme_class"] == "E3").sum()),
        "n_model_ready_DUB": int((model_ready["enzyme_class"] == "DUB").sum()),
        "n_required_enzymes_model": len(required_enzymes_model),
        "n_required_substrates_model": len(required_substrates_model),
        "n_required_accessions_model": len(required_accessions_model),
    }]).to_csv(
        QC_DIR / "model_ready_pairs_qc.csv",
        index=False
    )
    
    print("Saved model-ready pair tables.")
    print("Model-ready pairs:", model_ready.shape)
    print("Required model-ready accessions:", required_accessions_model.shape)

else:
    print("Do not save. Duplicate or label conflict detected.")

بعد از این FASTA coverage را باید روی فایل model-ready اجرا کنیم

در سلول FASTA coverage، به‌جای این:

REQ_ACCESSIONS_PATH = PAIR_DIR / "required_accessions_final.csv"

REQ_ENZYMES_PATH = PAIR_DIR / "required_enzymes_final.csv"

REQ_SUBSTRATES_PATH = PAIR_DIR / "required_substrates_final.csv"

از این‌ها استفاده کن:

REQ_ACCESSIONS_PATH = PAIR_DIR / "required_accessions_model_ready.csv"

REQ_ENZYMES_PATH = PAIR_DIR / "required_enzymes_model_ready.csv"

REQ_SUBSTRATES_PATH = PAIR_DIR / "required_substrates_model_ready.csv"

بعد سلول‌های FASTA coverage را دوباره از اول تا QC اجرا کن

-----

سلول ۱ — مسیرها و فایل‌های ورودی

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

PROJECT_ROOT = Path(".")

PAIR_DIR = PROJECT_ROOT / "Data_proc" / "pairs"
RAW_FASTA_DIR = PROJECT_ROOT / "Data_raw" / "uniprot" / "uniprot_fasta"
INTERIM_UNIPROT_DIR = PROJECT_ROOT / "Data_interim" / "uniprot"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

for d in [INTERIM_UNIPROT_DIR, QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PAIRS_ALL_PATH = PAIR_DIR / "pairs_all_final.csv"
REQ_ACCESSIONS_PATH = PAIR_DIR / "required_accessions_model_ready.csv"
REQ_ENZYMES_PATH = PAIR_DIR / "required_enzymes_model_ready.csv"
REQ_SUBSTRATES_PATH = PAIR_DIR / "required_substrates_model_ready.csv"

print("pairs_all exists:", PAIRS_ALL_PATH.exists(), PAIRS_ALL_PATH)
print("required_accessions exists:", REQ_ACCESSIONS_PATH.exists(), REQ_ACCESSIONS_PATH)
print("required_enzymes exists:", REQ_ENZYMES_PATH.exists(), REQ_ENZYMES_PATH)
print("required_substrates exists:", REQ_SUBSTRATES_PATH.exists(), REQ_SUBSTRATES_PATH)
print("raw fasta dir exists:", RAW_FASTA_DIR.exists(), RAW_FASTA_DIR)

pairs_all = pd.read_csv(PAIRS_ALL_PATH, dtype=str, low_memory=False)
required_accessions = pd.read_csv(REQ_ACCESSIONS_PATH, dtype=str, low_memory=False)
required_enzymes = pd.read_csv(REQ_ENZYMES_PATH, dtype=str, low_memory=False)
required_substrates = pd.read_csv(REQ_SUBSTRATES_PATH, dtype=str, low_memory=False)

print("pairs_all:", pairs_all.shape)
print("required_accessions:", required_accessions.shape)
print("required_enzymes:", required_enzymes.shape)
print("required_substrates:", required_substrates.shape)

display(pairs_all.head())
display(required_accessions.head())

سلول ۲ — توابع خواندن و اعتبارسنجی FASTA

In [ ]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWYBXZUOJ*-")

def normalize_ac(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return np.nan
    x = re.split(r"[;,|\s]", x)[0].strip()
    x = re.sub(r"-\d+$", "", x)
    return x.upper()


def extract_ac_from_header(header: str):
    """
    UniProt FASTA header usually:
    >sp|P12345|NAME_HUMAN ...
    """
    if not isinstance(header, str):
        return np.nan
    
    h = header.strip()
    if h.startswith(">"):
        h = h[1:].strip()
    
    if not h:
        return np.nan
    
    if "|" in h:
        parts = h.split("|")
        if len(parts) >= 2:
            return normalize_ac(parts[1])
    
    return normalize_ac(h.split()[0])


def read_single_fasta(path: Path):
    """
    Read one fasta file.
    Returns dict with status.
    """
    try:
        text = path.read_text(errors="ignore")
    except Exception as e:
        return {
            "ok": False,
            "error": f"read_error: {e}",
            "header": "",
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    if not text.strip():
        return {
            "ok": False,
            "error": "empty_file",
            "header": "",
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    
    if not lines:
        return {
            "ok": False,
            "error": "no_nonempty_lines",
            "header": "",
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    if not lines[0].startswith(">"):
        return {
            "ok": False,
            "error": "missing_header",
            "header": lines[0][:100],
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    header = lines[0]
    seq = "".join(lines[1:]).replace(" ", "").replace("\t", "").upper()
    
    if not seq:
        return {
            "ok": False,
            "error": "empty_sequence",
            "header": header,
            "seq": "",
            "header_ac": extract_ac_from_header(header),
            "length": 0,
        }
    
    invalid_chars = sorted(set(seq) - VALID_AA)
    
    if invalid_chars:
        return {
            "ok": False,
            "error": "invalid_amino_acids:" + "".join(invalid_chars),
            "header": header,
            "seq": seq,
            "header_ac": extract_ac_from_header(header),
            "length": len(seq),
        }
    
    return {
        "ok": True,
        "error": "",
        "header": header,
        "seq": seq,
        "header_ac": extract_ac_from_header(header),
        "length": len(seq),
    }

سلول ۳ — ساخت index از FASTAهای موجود

In [ ]:
# Detect all single FASTA files
fasta_files = sorted(
    list(RAW_FASTA_DIR.glob("*.fasta")) +
    list(RAW_FASTA_DIR.glob("*.fa")) +
    list(RAW_FASTA_DIR.glob("*.faa"))
)

print("FASTA files found:", len(fasta_files))

index_rows = []

for i, path in enumerate(fasta_files):
    if i % 1000 == 0:
        print("processing", i, "/", len(fasta_files))
    
    file_stem_ac = normalize_ac(path.stem)
    rec = read_single_fasta(path)
    
    index_rows.append({
        "file_path": str(path),
        "file_name": path.name,
        "file_stem_ac": file_stem_ac,
        "header_ac": rec["header_ac"],
        "ok": rec["ok"],
        "error": rec["error"],
        "length": rec["length"],
        "header": rec["header"],
    })

fasta_index = pd.DataFrame(index_rows)

print("FASTA index:", fasta_index.shape)
print("OK FASTA:", int(fasta_index["ok"].sum()) if len(fasta_index) else 0)
print("Bad FASTA:", int((~fasta_index["ok"]).sum()) if len(fasta_index) else 0)

display(fasta_index.head())
display(fasta_index["error"].value_counts(dropna=False).head(20))

سلول ۴ — اتصال accessionهای موردنیاز به FASTA index

In [ ]:
# Normalize required accessions
if "accession" in required_accessions.columns:
    req_col = "accession"
elif "uniprot_ac" in required_accessions.columns:
    req_col = "uniprot_ac"
else:
    req_col = required_accessions.columns[0]

required_accessions["accession"] = required_accessions[req_col].map(normalize_ac)
required_accessions = (
    required_accessions[["accession"]]
    .dropna()
    .drop_duplicates()
    .sort_values("accession")
    .reset_index(drop=True)
)

# Prefer file_stem_ac matching; header_ac is secondary evidence
ok_index = fasta_index[fasta_index["ok"]].copy()

# If duplicate FASTA files for same AC exist, keep first
ok_by_stem = (
    ok_index
    .dropna(subset=["file_stem_ac"])
    .drop_duplicates(subset=["file_stem_ac"], keep="first")
    .set_index("file_stem_ac")
)

coverage_rows = []

for acc in required_accessions["accession"]:
    if acc in ok_by_stem.index:
        row = ok_by_stem.loc[acc]
        coverage_rows.append({
            "accession": acc,
            "has_fasta": True,
            "fasta_ok": True,
            "file_path": row["file_path"],
            "file_name": row["file_name"],
            "length": int(row["length"]),
            "header_ac": row["header_ac"],
            "error": "",
        })
    else:
        # Check if present but bad
        bad_match = fasta_index[fasta_index["file_stem_ac"] == acc]
        if len(bad_match):
            err = ";".join(bad_match["error"].astype(str).unique())
            coverage_rows.append({
                "accession": acc,
                "has_fasta": True,
                "fasta_ok": False,
                "file_path": bad_match.iloc[0]["file_path"],
                "file_name": bad_match.iloc[0]["file_name"],
                "length": int(bad_match.iloc[0]["length"]),
                "header_ac": bad_match.iloc[0]["header_ac"],
                "error": err,
            })
        else:
            coverage_rows.append({
                "accession": acc,
                "has_fasta": False,
                "fasta_ok": False,
                "file_path": "",
                "file_name": "",
                "length": 0,
                "header_ac": "",
                "error": "missing_file",
            })

fasta_coverage = pd.DataFrame(coverage_rows)

missing_required = fasta_coverage[~fasta_coverage["has_fasta"]].copy()
bad_required = fasta_coverage[
    fasta_coverage["has_fasta"] & (~fasta_coverage["fasta_ok"])
].copy()

print("Required accessions:", len(fasta_coverage))
print("Covered OK:", int(fasta_coverage["fasta_ok"].sum()))
print("Missing:", len(missing_required))
print("Bad:", len(bad_required))

display(fasta_coverage.head())
display(missing_required.head())
display(bad_required.head())

سلول ۵ — QC بر اساس نقش enzyme/substrate

In [ ]:
def prepare_role_df(df, col_guess):
    out = df.copy()
    if col_guess in out.columns:
        col = col_guess
    elif "accession" in out.columns:
        col = "accession"
    elif "uniprot_ac" in out.columns:
        col = "uniprot_ac"
    else:
        col = out.columns[0]
    
    out["accession"] = out[col].map(normalize_ac)
    out = out[["accession"]].dropna().drop_duplicates()
    return out


enz_req = prepare_role_df(required_enzymes, "enz_ac")
sub_req = prepare_role_df(required_substrates, "sub_ac")

enz_cov = enz_req.merge(fasta_coverage, on="accession", how="left")
sub_cov = sub_req.merge(fasta_coverage, on="accession", how="left")

fasta_coverage_qc = pd.DataFrame([
    {
        "dataset": "all_required_accessions",
        "n_required": len(fasta_coverage),
        "n_fasta_ok": int(fasta_coverage["fasta_ok"].sum()),
        "n_missing": int((~fasta_coverage["has_fasta"]).sum()),
        "n_bad": int((fasta_coverage["has_fasta"] & ~fasta_coverage["fasta_ok"]).sum()),
        "coverage_fraction": float(fasta_coverage["fasta_ok"].mean()) if len(fasta_coverage) else np.nan,
        "mean_length_ok": float(fasta_coverage.loc[fasta_coverage["fasta_ok"], "length"].mean()) if fasta_coverage["fasta_ok"].any() else np.nan,
        "median_length_ok": float(fasta_coverage.loc[fasta_coverage["fasta_ok"], "length"].median()) if fasta_coverage["fasta_ok"].any() else np.nan,
    },
    {
        "dataset": "required_enzymes",
        "n_required": len(enz_cov),
        "n_fasta_ok": int(enz_cov["fasta_ok"].fillna(False).sum()),
        "n_missing": int((enz_cov["has_fasta"].fillna(False) == False).sum()),
        "n_bad": int((enz_cov["has_fasta"].fillna(False) & (enz_cov["fasta_ok"].fillna(False) == False)).sum()),
        "coverage_fraction": float(enz_cov["fasta_ok"].fillna(False).mean()) if len(enz_cov) else np.nan,
        "mean_length_ok": float(enz_cov.loc[enz_cov["fasta_ok"] == True, "length"].mean()) if (enz_cov["fasta_ok"] == True).any() else np.nan,
        "median_length_ok": float(enz_cov.loc[enz_cov["fasta_ok"] == True, "length"].median()) if (enz_cov["fasta_ok"] == True).any() else np.nan,
    },
    {
        "dataset": "required_substrates",
        "n_required": len(sub_cov),
        "n_fasta_ok": int(sub_cov["fasta_ok"].fillna(False).sum()),
        "n_missing": int((sub_cov["has_fasta"].fillna(False) == False).sum()),
        "n_bad": int((sub_cov["has_fasta"].fillna(False) & (sub_cov["fasta_ok"].fillna(False) == False)).sum()),
        "coverage_fraction": float(sub_cov["fasta_ok"].fillna(False).mean()) if len(sub_cov) else np.nan,
        "mean_length_ok": float(sub_cov.loc[sub_cov["fasta_ok"] == True, "length"].mean()) if (sub_cov["fasta_ok"] == True).any() else np.nan,
        "median_length_ok": float(sub_cov.loc[sub_cov["fasta_ok"] == True, "length"].median()) if (sub_cov["fasta_ok"] == True).any() else np.nan,
    },
])

display(fasta_coverage_qc)

الان این سلول را اجرا کن تا ۴ فایل خراب را ببینیم

In [ ]:
bad_fasta_files = fasta_index[~fasta_index["ok"]].copy()

print("Bad FASTA files:", len(bad_fasta_files))
display(bad_fasta_files[[
    "file_name",
    "file_stem_ac",
    "header_ac",
    "error",
    "length",
    "file_path",
    "header"
]])

for _, row in bad_fasta_files.iterrows():
    path = Path(row["file_path"])
    print("\n" + "="*100)
    print("FILE:", row["file_name"])
    print("ERROR:", row["error"])
    print("PATH:", path)
    
    try:
        text = path.read_text(errors="replace")
        print("\nFirst 1000 characters:")
        print(text[:1000])
    except Exception as e:
        print("Could not read:", e)

قدم بعدی: index را فقط با single FASTA بساز

In [ ]:
fasta_files = sorted(
    list(RAW_FASTA_DIR.glob("*.fasta")) +
    list(RAW_FASTA_DIR.glob("*.fa")) +
    list(RAW_FASTA_DIR.glob("*.faa"))
)

# فقط فایل‌هایی که stem آن‌ها شبیه accession واقعی است
EXCLUDE_FASTA_NAMES = {
    "all_sequences.fasta",
    "enzymes.fasta",
    "substrates.fasta",
    "combined_required_sequences.fasta",
    "required_sequences_final.fasta",
    "enzymes_final.fasta",
    "substrates_final.fasta",
}

fasta_files = [
    p for p in fasta_files
    if p.name not in EXCLUDE_FASTA_NAMES
]

print("Single FASTA files found:", len(fasta_files))

index_rows = []

for i, path in enumerate(fasta_files):
    if i % 1000 == 0:
        print("processing", i, "/", len(fasta_files))
    
    file_stem_ac = normalize_ac(path.stem)
    rec = read_single_fasta(path)
    
    index_rows.append({
        "file_path": str(path),
        "file_name": path.name,
        "file_stem_ac": file_stem_ac,
        "header_ac": rec["header_ac"],
        "ok": rec["ok"],
        "error": rec["error"],
        "length": rec["length"],
        "header": rec["header"],
    })

fasta_index = pd.DataFrame(index_rows)

print("FASTA index:", fasta_index.shape)
print("OK FASTA:", int(fasta_index["ok"].sum()) if len(fasta_index) else 0)
print("Bad FASTA:", int((~fasta_index["ok"]).sum()) if len(fasta_index) else 0)

display(fasta_index.head())
display(fasta_index["error"].value_counts(dropna=False).head(20))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

PROJECT_ROOT = Path(".")

PAIR_DIR = PROJECT_ROOT / "Data_proc" / "pairs"
RAW_FASTA_DIR = PROJECT_ROOT / "Data_raw" / "uniprot" / "uniprot_fasta"
INTERIM_UNIPROT_DIR = PROJECT_ROOT / "Data_interim" / "uniprot"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

for d in [INTERIM_UNIPROT_DIR, QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PAIRS_ALL_PATH = PAIR_DIR / "pairs_all_final.csv"
REQ_ACCESSIONS_PATH = PAIR_DIR / "required_accessions_model_ready.csv"
REQ_ENZYMES_PATH = PAIR_DIR / "required_enzymes_model_ready.csv"
REQ_SUBSTRATES_PATH = PAIR_DIR / "required_substrates_model_ready.csv"

print("pairs_all exists:", PAIRS_ALL_PATH.exists(), PAIRS_ALL_PATH)
print("required_accessions exists:", REQ_ACCESSIONS_PATH.exists(), REQ_ACCESSIONS_PATH)
print("required_enzymes exists:", REQ_ENZYMES_PATH.exists(), REQ_ENZYMES_PATH)
print("required_substrates exists:", REQ_SUBSTRATES_PATH.exists(), REQ_SUBSTRATES_PATH)
print("raw fasta dir exists:", RAW_FASTA_DIR.exists(), RAW_FASTA_DIR)

pairs_all = pd.read_csv(PAIRS_ALL_PATH, dtype=str, low_memory=False)
required_accessions = pd.read_csv(REQ_ACCESSIONS_PATH, dtype=str, low_memory=False)
required_enzymes = pd.read_csv(REQ_ENZYMES_PATH, dtype=str, low_memory=False)
required_substrates = pd.read_csv(REQ_SUBSTRATES_PATH, dtype=str, low_memory=False)

print("pairs_all:", pairs_all.shape)
print("required_accessions:", required_accessions.shape)
print("required_enzymes:", required_enzymes.shape)
print("required_substrates:", required_substrates.shape)

display(pairs_all.head())
display(required_accessions.head())

In [ ]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWYBXZUOJ*-")

def normalize_ac(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return np.nan
    x = re.split(r"[;,|\s]", x)[0].strip()
    x = re.sub(r"-\d+$", "", x)
    return x.upper()


def extract_ac_from_header(header: str):
    """
    UniProt FASTA header usually:
    >sp|P12345|NAME_HUMAN ...
    """
    if not isinstance(header, str):
        return np.nan
    
    h = header.strip()
    if h.startswith(">"):
        h = h[1:].strip()
    
    if not h:
        return np.nan
    
    if "|" in h:
        parts = h.split("|")
        if len(parts) >= 2:
            return normalize_ac(parts[1])
    
    return normalize_ac(h.split()[0])


def read_single_fasta(path: Path):
    """
    Read one fasta file.
    Returns dict with status.
    """
    try:
        text = path.read_text(errors="ignore")
    except Exception as e:
        return {
            "ok": False,
            "error": f"read_error: {e}",
            "header": "",
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    if not text.strip():
        return {
            "ok": False,
            "error": "empty_file",
            "header": "",
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    
    if not lines:
        return {
            "ok": False,
            "error": "no_nonempty_lines",
            "header": "",
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    if not lines[0].startswith(">"):
        return {
            "ok": False,
            "error": "missing_header",
            "header": lines[0][:100],
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    header = lines[0]
    seq = "".join(lines[1:]).replace(" ", "").replace("\t", "").upper()
    
    if not seq:
        return {
            "ok": False,
            "error": "empty_sequence",
            "header": header,
            "seq": "",
            "header_ac": extract_ac_from_header(header),
            "length": 0,
        }
    
    invalid_chars = sorted(set(seq) - VALID_AA)
    
    if invalid_chars:
        return {
            "ok": False,
            "error": "invalid_amino_acids:" + "".join(invalid_chars),
            "header": header,
            "seq": seq,
            "header_ac": extract_ac_from_header(header),
            "length": len(seq),
        }
    
    return {
        "ok": True,
        "error": "",
        "header": header,
        "seq": seq,
        "header_ac": extract_ac_from_header(header),
        "length": len(seq),
    }

In [ ]:
# Detect all single FASTA files
fasta_files = sorted(
    list(RAW_FASTA_DIR.glob("*.fasta")) +
    list(RAW_FASTA_DIR.glob("*.fa")) +
    list(RAW_FASTA_DIR.glob("*.faa"))
)

print("FASTA files found:", len(fasta_files))

index_rows = []

for i, path in enumerate(fasta_files):
    if i % 1000 == 0:
        print("processing", i, "/", len(fasta_files))
    
    file_stem_ac = normalize_ac(path.stem)
    rec = read_single_fasta(path)
    
    index_rows.append({
        "file_path": str(path),
        "file_name": path.name,
        "file_stem_ac": file_stem_ac,
        "header_ac": rec["header_ac"],
        "ok": rec["ok"],
        "error": rec["error"],
        "length": rec["length"],
        "header": rec["header"],
    })

fasta_index = pd.DataFrame(index_rows)

print("FASTA index:", fasta_index.shape)
print("OK FASTA:", int(fasta_index["ok"].sum()) if len(fasta_index) else 0)
print("Bad FASTA:", int((~fasta_index["ok"]).sum()) if len(fasta_index) else 0)

display(fasta_index.head())
display(fasta_index["error"].value_counts(dropna=False).head(20))

In [ ]:
# Normalize required accessions
if "accession" in required_accessions.columns:
    req_col = "accession"
elif "uniprot_ac" in required_accessions.columns:
    req_col = "uniprot_ac"
else:
    req_col = required_accessions.columns[0]

required_accessions["accession"] = required_accessions[req_col].map(normalize_ac)
required_accessions = (
    required_accessions[["accession"]]
    .dropna()
    .drop_duplicates()
    .sort_values("accession")
    .reset_index(drop=True)
)

# Prefer file_stem_ac matching; header_ac is secondary evidence
ok_index = fasta_index[fasta_index["ok"]].copy()

# If duplicate FASTA files for same AC exist, keep first
ok_by_stem = (
    ok_index
    .dropna(subset=["file_stem_ac"])
    .drop_duplicates(subset=["file_stem_ac"], keep="first")
    .set_index("file_stem_ac")
)

coverage_rows = []

for acc in required_accessions["accession"]:
    if acc in ok_by_stem.index:
        row = ok_by_stem.loc[acc]
        coverage_rows.append({
            "accession": acc,
            "has_fasta": True,
            "fasta_ok": True,
            "file_path": row["file_path"],
            "file_name": row["file_name"],
            "length": int(row["length"]),
            "header_ac": row["header_ac"],
            "error": "",
        })
    else:
        # Check if present but bad
        bad_match = fasta_index[fasta_index["file_stem_ac"] == acc]
        if len(bad_match):
            err = ";".join(bad_match["error"].astype(str).unique())
            coverage_rows.append({
                "accession": acc,
                "has_fasta": True,
                "fasta_ok": False,
                "file_path": bad_match.iloc[0]["file_path"],
                "file_name": bad_match.iloc[0]["file_name"],
                "length": int(bad_match.iloc[0]["length"]),
                "header_ac": bad_match.iloc[0]["header_ac"],
                "error": err,
            })
        else:
            coverage_rows.append({
                "accession": acc,
                "has_fasta": False,
                "fasta_ok": False,
                "file_path": "",
                "file_name": "",
                "length": 0,
                "header_ac": "",
                "error": "missing_file",
            })

fasta_coverage = pd.DataFrame(coverage_rows)

missing_required = fasta_coverage[~fasta_coverage["has_fasta"]].copy()
bad_required = fasta_coverage[
    fasta_coverage["has_fasta"] & (~fasta_coverage["fasta_ok"])
].copy()

print("Required accessions:", len(fasta_coverage))
print("Covered OK:", int(fasta_coverage["fasta_ok"].sum()))
print("Missing:", len(missing_required))
print("Bad:", len(bad_required))

display(fasta_coverage.head())
display(missing_required.head())
display(bad_required.head())

In [ ]:
def prepare_role_df(df, col_guess):
    out = df.copy()
    if col_guess in out.columns:
        col = col_guess
    elif "accession" in out.columns:
        col = "accession"
    elif "uniprot_ac" in out.columns:
        col = "uniprot_ac"
    else:
        col = out.columns[0]
    
    out["accession"] = out[col].map(normalize_ac)
    out = out[["accession"]].dropna().drop_duplicates()
    return out


enz_req = prepare_role_df(required_enzymes, "enz_ac")
sub_req = prepare_role_df(required_substrates, "sub_ac")

enz_cov = enz_req.merge(fasta_coverage, on="accession", how="left")
sub_cov = sub_req.merge(fasta_coverage, on="accession", how="left")

fasta_coverage_qc = pd.DataFrame([
    {
        "dataset": "all_required_accessions",
        "n_required": len(fasta_coverage),
        "n_fasta_ok": int(fasta_coverage["fasta_ok"].sum()),
        "n_missing": int((~fasta_coverage["has_fasta"]).sum()),
        "n_bad": int((fasta_coverage["has_fasta"] & ~fasta_coverage["fasta_ok"]).sum()),
        "coverage_fraction": float(fasta_coverage["fasta_ok"].mean()) if len(fasta_coverage) else np.nan,
        "mean_length_ok": float(fasta_coverage.loc[fasta_coverage["fasta_ok"], "length"].mean()) if fasta_coverage["fasta_ok"].any() else np.nan,
        "median_length_ok": float(fasta_coverage.loc[fasta_coverage["fasta_ok"], "length"].median()) if fasta_coverage["fasta_ok"].any() else np.nan,
    },
    {
        "dataset": "required_enzymes",
        "n_required": len(enz_cov),
        "n_fasta_ok": int(enz_cov["fasta_ok"].fillna(False).sum()),
        "n_missing": int((enz_cov["has_fasta"].fillna(False) == False).sum()),
        "n_bad": int((enz_cov["has_fasta"].fillna(False) & (enz_cov["fasta_ok"].fillna(False) == False)).sum()),
        "coverage_fraction": float(enz_cov["fasta_ok"].fillna(False).mean()) if len(enz_cov) else np.nan,
        "mean_length_ok": float(enz_cov.loc[enz_cov["fasta_ok"] == True, "length"].mean()) if (enz_cov["fasta_ok"] == True).any() else np.nan,
        "median_length_ok": float(enz_cov.loc[enz_cov["fasta_ok"] == True, "length"].median()) if (enz_cov["fasta_ok"] == True).any() else np.nan,
    },
    {
        "dataset": "required_substrates",
        "n_required": len(sub_cov),
        "n_fasta_ok": int(sub_cov["fasta_ok"].fillna(False).sum()),
        "n_missing": int((sub_cov["has_fasta"].fillna(False) == False).sum()),
        "n_bad": int((sub_cov["has_fasta"].fillna(False) & (sub_cov["fasta_ok"].fillna(False) == False)).sum()),
        "coverage_fraction": float(sub_cov["fasta_ok"].fillna(False).mean()) if len(sub_cov) else np.nan,
        "mean_length_ok": float(sub_cov.loc[sub_cov["fasta_ok"] == True, "length"].mean()) if (sub_cov["fasta_ok"] == True).any() else np.nan,
        "median_length_ok": float(sub_cov.loc[sub_cov["fasta_ok"] == True, "length"].median()) if (sub_cov["fasta_ok"] == True).any() else np.nan,
    },
])

display(fasta_coverage_qc)

قدم بعدی اصلی: missingها و P08107 را دانلود کنیم

In [ ]:
import time
import requests
from pathlib import Path
import pandas as pd
import numpy as np

# missing + bad واقعی از coverage model-ready
to_download = pd.concat([
    missing_required[["accession"]],
    bad_required[["accession"]],
], ignore_index=True)

to_download["accession"] = to_download["accession"].map(normalize_ac)
to_download = (
    to_download
    .dropna()
    .drop_duplicates()
    .sort_values("accession")
    .reset_index(drop=True)
)

print("Accessions to download:", len(to_download))
display(to_download.head(50))

download_log_rows = []

def is_valid_fasta_text(text: str, expected_acc: str):
    if not isinstance(text, str) or not text.strip():
        return False, "empty_response"
    
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    
    if not lines or not lines[0].startswith(">"):
        return False, "missing_fasta_header"
    
    seq = "".join(lines[1:]).replace(" ", "").replace("\t", "").upper()
    
    if not seq:
        return False, "empty_sequence"
    
    invalid = sorted(set(seq) - VALID_AA)
    if invalid:
        return False, "invalid_amino_acids:" + "".join(invalid)
    
    return True, ""

session = requests.Session()

for i, acc in enumerate(to_download["accession"]):
    url = f"https://rest.uniprot.org/uniprotkb/{acc}.fasta"
    out_path = RAW_FASTA_DIR / f"{acc}.fasta"
    
    print(f"{i+1}/{len(to_download)} downloading {acc}")
    
    try:
        r = session.get(url, timeout=30)
        status_code = r.status_code
        text = r.text
        
        if status_code == 200:
            ok, err = is_valid_fasta_text(text, acc)
            
            if ok:
                out_path.write_text(text)
                download_log_rows.append({
                    "accession": acc,
                    "status": "downloaded",
                    "http_status": status_code,
                    "file_path": str(out_path),
                    "error": "",
                })
            else:
                download_log_rows.append({
                    "accession": acc,
                    "status": "invalid_response",
                    "http_status": status_code,
                    "file_path": "",
                    "error": err,
                })
        else:
            download_log_rows.append({
                "accession": acc,
                "status": "http_error",
                "http_status": status_code,
                "file_path": "",
                "error": text[:300],
            })
    
    except Exception as e:
        download_log_rows.append({
            "accession": acc,
            "status": "exception",
            "http_status": "",
            "file_path": "",
            "error": str(e),
        })
    
    time.sleep(0.2)

download_log = pd.DataFrame(download_log_rows)

display(download_log["status"].value_counts(dropna=False))
display(download_log.head())

download_log.to_csv(
    QC_DIR / "missing_bad_fasta_download_log.csv",
    index=False
)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

PROJECT_ROOT = Path(".")

PAIR_DIR = PROJECT_ROOT / "Data_proc" / "pairs"
RAW_FASTA_DIR = PROJECT_ROOT / "Data_raw" / "uniprot" / "uniprot_fasta"
INTERIM_UNIPROT_DIR = PROJECT_ROOT / "Data_interim" / "uniprot"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

for d in [INTERIM_UNIPROT_DIR, QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PAIRS_ALL_PATH = PAIR_DIR / "pairs_all_final.csv"
REQ_ACCESSIONS_PATH = PAIR_DIR / "required_accessions_model_ready.csv"
REQ_ENZYMES_PATH = PAIR_DIR / "required_enzymes_model_ready.csv"
REQ_SUBSTRATES_PATH = PAIR_DIR / "required_substrates_model_ready.csv"

print("pairs_all exists:", PAIRS_ALL_PATH.exists(), PAIRS_ALL_PATH)
print("required_accessions exists:", REQ_ACCESSIONS_PATH.exists(), REQ_ACCESSIONS_PATH)
print("required_enzymes exists:", REQ_ENZYMES_PATH.exists(), REQ_ENZYMES_PATH)
print("required_substrates exists:", REQ_SUBSTRATES_PATH.exists(), REQ_SUBSTRATES_PATH)
print("raw fasta dir exists:", RAW_FASTA_DIR.exists(), RAW_FASTA_DIR)

pairs_all = pd.read_csv(PAIRS_ALL_PATH, dtype=str, low_memory=False)
required_accessions = pd.read_csv(REQ_ACCESSIONS_PATH, dtype=str, low_memory=False)
required_enzymes = pd.read_csv(REQ_ENZYMES_PATH, dtype=str, low_memory=False)
required_substrates = pd.read_csv(REQ_SUBSTRATES_PATH, dtype=str, low_memory=False)

print("pairs_all:", pairs_all.shape)
print("required_accessions:", required_accessions.shape)
print("required_enzymes:", required_enzymes.shape)
print("required_substrates:", required_substrates.shape)

display(pairs_all.head())
display(required_accessions.head())

In [ ]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWYBXZUOJ*-")

def normalize_ac(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return np.nan
    x = re.split(r"[;,|\s]", x)[0].strip()
    x = re.sub(r"-\d+$", "", x)
    return x.upper()


def extract_ac_from_header(header: str):
    """
    UniProt FASTA header usually:
    >sp|P12345|NAME_HUMAN ...
    """
    if not isinstance(header, str):
        return np.nan
    
    h = header.strip()
    if h.startswith(">"):
        h = h[1:].strip()
    
    if not h:
        return np.nan
    
    if "|" in h:
        parts = h.split("|")
        if len(parts) >= 2:
            return normalize_ac(parts[1])
    
    return normalize_ac(h.split()[0])


def read_single_fasta(path: Path):
    """
    Read one fasta file.
    Returns dict with status.
    """
    try:
        text = path.read_text(errors="ignore")
    except Exception as e:
        return {
            "ok": False,
            "error": f"read_error: {e}",
            "header": "",
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    if not text.strip():
        return {
            "ok": False,
            "error": "empty_file",
            "header": "",
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    
    if not lines:
        return {
            "ok": False,
            "error": "no_nonempty_lines",
            "header": "",
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    if not lines[0].startswith(">"):
        return {
            "ok": False,
            "error": "missing_header",
            "header": lines[0][:100],
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    header = lines[0]
    seq = "".join(lines[1:]).replace(" ", "").replace("\t", "").upper()
    
    if not seq:
        return {
            "ok": False,
            "error": "empty_sequence",
            "header": header,
            "seq": "",
            "header_ac": extract_ac_from_header(header),
            "length": 0,
        }
    
    invalid_chars = sorted(set(seq) - VALID_AA)
    
    if invalid_chars:
        return {
            "ok": False,
            "error": "invalid_amino_acids:" + "".join(invalid_chars),
            "header": header,
            "seq": seq,
            "header_ac": extract_ac_from_header(header),
            "length": len(seq),
        }
    
    return {
        "ok": True,
        "error": "",
        "header": header,
        "seq": seq,
        "header_ac": extract_ac_from_header(header),
        "length": len(seq),
    }

In [ ]:
# Detect all single FASTA files
fasta_files = sorted(
    list(RAW_FASTA_DIR.glob("*.fasta")) +
    list(RAW_FASTA_DIR.glob("*.fa")) +
    list(RAW_FASTA_DIR.glob("*.faa"))
)

print("FASTA files found:", len(fasta_files))

index_rows = []

for i, path in enumerate(fasta_files):
    if i % 1000 == 0:
        print("processing", i, "/", len(fasta_files))
    
    file_stem_ac = normalize_ac(path.stem)
    rec = read_single_fasta(path)
    
    index_rows.append({
        "file_path": str(path),
        "file_name": path.name,
        "file_stem_ac": file_stem_ac,
        "header_ac": rec["header_ac"],
        "ok": rec["ok"],
        "error": rec["error"],
        "length": rec["length"],
        "header": rec["header"],
    })

fasta_index = pd.DataFrame(index_rows)

print("FASTA index:", fasta_index.shape)
print("OK FASTA:", int(fasta_index["ok"].sum()) if len(fasta_index) else 0)
print("Bad FASTA:", int((~fasta_index["ok"]).sum()) if len(fasta_index) else 0)

display(fasta_index.head())
display(fasta_index["error"].value_counts(dropna=False).head(20))

In [ ]:
# Normalize required accessions
if "accession" in required_accessions.columns:
    req_col = "accession"
elif "uniprot_ac" in required_accessions.columns:
    req_col = "uniprot_ac"
else:
    req_col = required_accessions.columns[0]

required_accessions["accession"] = required_accessions[req_col].map(normalize_ac)
required_accessions = (
    required_accessions[["accession"]]
    .dropna()
    .drop_duplicates()
    .sort_values("accession")
    .reset_index(drop=True)
)

# Prefer file_stem_ac matching; header_ac is secondary evidence
ok_index = fasta_index[fasta_index["ok"]].copy()

# If duplicate FASTA files for same AC exist, keep first
ok_by_stem = (
    ok_index
    .dropna(subset=["file_stem_ac"])
    .drop_duplicates(subset=["file_stem_ac"], keep="first")
    .set_index("file_stem_ac")
)

coverage_rows = []

for acc in required_accessions["accession"]:
    if acc in ok_by_stem.index:
        row = ok_by_stem.loc[acc]
        coverage_rows.append({
            "accession": acc,
            "has_fasta": True,
            "fasta_ok": True,
            "file_path": row["file_path"],
            "file_name": row["file_name"],
            "length": int(row["length"]),
            "header_ac": row["header_ac"],
            "error": "",
        })
    else:
        # Check if present but bad
        bad_match = fasta_index[fasta_index["file_stem_ac"] == acc]
        if len(bad_match):
            err = ";".join(bad_match["error"].astype(str).unique())
            coverage_rows.append({
                "accession": acc,
                "has_fasta": True,
                "fasta_ok": False,
                "file_path": bad_match.iloc[0]["file_path"],
                "file_name": bad_match.iloc[0]["file_name"],
                "length": int(bad_match.iloc[0]["length"]),
                "header_ac": bad_match.iloc[0]["header_ac"],
                "error": err,
            })
        else:
            coverage_rows.append({
                "accession": acc,
                "has_fasta": False,
                "fasta_ok": False,
                "file_path": "",
                "file_name": "",
                "length": 0,
                "header_ac": "",
                "error": "missing_file",
            })

fasta_coverage = pd.DataFrame(coverage_rows)

missing_required = fasta_coverage[~fasta_coverage["has_fasta"]].copy()
bad_required = fasta_coverage[
    fasta_coverage["has_fasta"] & (~fasta_coverage["fasta_ok"])
].copy()

print("Required accessions:", len(fasta_coverage))
print("Covered OK:", int(fasta_coverage["fasta_ok"].sum()))
print("Missing:", len(missing_required))
print("Bad:", len(bad_required))

display(fasta_coverage.head())
display(missing_required.head())
display(bad_required.head())

In [ ]:
def prepare_role_df(df, col_guess):
    out = df.copy()
    if col_guess in out.columns:
        col = col_guess
    elif "accession" in out.columns:
        col = "accession"
    elif "uniprot_ac" in out.columns:
        col = "uniprot_ac"
    else:
        col = out.columns[0]
    
    out["accession"] = out[col].map(normalize_ac)
    out = out[["accession"]].dropna().drop_duplicates()
    return out


enz_req = prepare_role_df(required_enzymes, "enz_ac")
sub_req = prepare_role_df(required_substrates, "sub_ac")

enz_cov = enz_req.merge(fasta_coverage, on="accession", how="left")
sub_cov = sub_req.merge(fasta_coverage, on="accession", how="left")

fasta_coverage_qc = pd.DataFrame([
    {
        "dataset": "all_required_accessions",
        "n_required": len(fasta_coverage),
        "n_fasta_ok": int(fasta_coverage["fasta_ok"].sum()),
        "n_missing": int((~fasta_coverage["has_fasta"]).sum()),
        "n_bad": int((fasta_coverage["has_fasta"] & ~fasta_coverage["fasta_ok"]).sum()),
        "coverage_fraction": float(fasta_coverage["fasta_ok"].mean()) if len(fasta_coverage) else np.nan,
        "mean_length_ok": float(fasta_coverage.loc[fasta_coverage["fasta_ok"], "length"].mean()) if fasta_coverage["fasta_ok"].any() else np.nan,
        "median_length_ok": float(fasta_coverage.loc[fasta_coverage["fasta_ok"], "length"].median()) if fasta_coverage["fasta_ok"].any() else np.nan,
    },
    {
        "dataset": "required_enzymes",
        "n_required": len(enz_cov),
        "n_fasta_ok": int(enz_cov["fasta_ok"].fillna(False).sum()),
        "n_missing": int((enz_cov["has_fasta"].fillna(False) == False).sum()),
        "n_bad": int((enz_cov["has_fasta"].fillna(False) & (enz_cov["fasta_ok"].fillna(False) == False)).sum()),
        "coverage_fraction": float(enz_cov["fasta_ok"].fillna(False).mean()) if len(enz_cov) else np.nan,
        "mean_length_ok": float(enz_cov.loc[enz_cov["fasta_ok"] == True, "length"].mean()) if (enz_cov["fasta_ok"] == True).any() else np.nan,
        "median_length_ok": float(enz_cov.loc[enz_cov["fasta_ok"] == True, "length"].median()) if (enz_cov["fasta_ok"] == True).any() else np.nan,
    },
    {
        "dataset": "required_substrates",
        "n_required": len(sub_cov),
        "n_fasta_ok": int(sub_cov["fasta_ok"].fillna(False).sum()),
        "n_missing": int((sub_cov["has_fasta"].fillna(False) == False).sum()),
        "n_bad": int((sub_cov["has_fasta"].fillna(False) & (sub_cov["fasta_ok"].fillna(False) == False)).sum()),
        "coverage_fraction": float(sub_cov["fasta_ok"].fillna(False).mean()) if len(sub_cov) else np.nan,
        "mean_length_ok": float(sub_cov.loc[sub_cov["fasta_ok"] == True, "length"].mean()) if (sub_cov["fasta_ok"] == True).any() else np.nan,
        "median_length_ok": float(sub_cov.loc[sub_cov["fasta_ok"] == True, "length"].median()) if (sub_cov["fasta_ok"] == True).any() else np.nan,
    },
])

display(fasta_coverage_qc)

عالی. الان تقریباً کامل شدی؛ فقط یک FASTA خراب مانده.

قدم بعدی: دقیقاً ببین bad accession چیست


In [ ]:
print("Bad required:")
display(bad_required)

print("Bad substrate:")
bad_sub = sub_cov[sub_cov["has_fasta"].fillna(False) & (sub_cov["fasta_ok"].fillna(False) == False)]
display(bad_sub)

print("Rows in model_ready using bad accession:")
bad_accs = set(bad_required["accession"].astype(str))

bad_pairs = model_ready[
    model_ready["enz_ac"].astype(str).isin(bad_accs) |
    model_ready["sub_ac"].astype(str).isin(bad_accs)
].copy()

print("bad pairs:", bad_pairs.shape)
display(bad_pairs[[
    "pair_id", "enzyme_class", "enz_ac", "sub_ac",
    "enz_gene", "sub_gene", "label", "source", "pmid"
]].head(100))

In [ ]:
import requests
from pathlib import Path

acc = "P08107"
url = f"https://rest.uniprot.org/uniprotkb/{acc}.fasta"
out_path = RAW_FASTA_DIR / f"{acc}.fasta"

r = requests.get(url, timeout=30)

print("HTTP status:", r.status_code)
print("Response first 300 chars:")
print(r.text[:300])

if r.status_code == 200 and r.text.strip().startswith(">"):
    out_path.write_text(r.text)
    print("Saved:", out_path)
else:
    print("Download failed or invalid response.")

۱. اول ببین P08107 کجا استفاده شده

In [ ]:
bad_acc = "P08107"

bad_pairs = model_ready[
    (model_ready["enz_ac"].astype(str) == bad_acc) |
    (model_ready["sub_ac"].astype(str) == bad_acc)
].copy()

print("Rows using P08107:", bad_pairs.shape)

display(bad_pairs[[
    "pair_id", "enzyme_class", "enz_ac", "sub_ac",
    "enz_gene", "sub_gene",
    "enzyme_type", "label", "source", "pmid"
]])

۲. تلاش دوم: query با accession و گرفتن FASTA از search

In [ ]:
import requests
from pathlib import Path

acc = "P08107"
url = "https://rest.uniprot.org/uniprotkb/search"

params = {
    "query": f"accession:{acc}",
    "format": "fasta",
    "size": 1,
}

r = requests.get(url, params=params, timeout=30)

print("URL:", r.url)
print("HTTP status:", r.status_code)
print("Response length:", len(r.text))
print("First 500 chars:")
print(r.text[:500])

if r.status_code == 200 and r.text.strip().startswith(">"):
    out_path = RAW_FASTA_DIR / f"{acc}.fasta"
    out_path.write_text(r.text)
    print("Saved:", out_path)
else:
    print("No valid FASTA returned.")

۳. اگر باز هم خالی بود: query با gene symbol

In [ ]:
gene = bad_pairs.loc[
    (bad_pairs["sub_ac"].astype(str) == "P08107"), "sub_gene"
].dropna().astype(str).iloc[0]

print("Gene for P08107:", gene)

url = "https://rest.uniprot.org/uniprotkb/search"

params = {
    "query": f"gene_exact:{gene} AND organism_id:9606",
    "format": "fasta",
    "size": 5,
}

r = requests.get(url, params=params, timeout=30)

print("URL:", r.url)
print("HTTP status:", r.status_code)
print("Response length:", len(r.text))
print("First 1000 chars:")
print(r.text[:1000])

۱. اول FASTA صحیح P0DMV8 را ذخیره کن

In [ ]:
import requests
from pathlib import Path

gene = "HSPA1A"

url = "https://rest.uniprot.org/uniprotkb/search"
params = {
    "query": f"gene_exact:{gene} AND organism_id:9606",
    "format": "fasta",
    "size": 5,
}

r = requests.get(url, params=params, timeout=30)

print("HTTP status:", r.status_code)
print("Response length:", len(r.text))
print(r.text[:300])

text = r.text.strip()

# Split multi-FASTA into records
records = []
current = []

for line in text.splitlines():
    if line.startswith(">") and current:
        records.append("\n".join(current))
        current = [line]
    else:
        current.append(line)

if current:
    records.append("\n".join(current))

print("Number of FASTA records:", len(records))

first_record = records[0].strip()
first_header = first_record.splitlines()[0]
new_acc = first_header.split("|")[1] if "|" in first_header else None

print("Selected accession:", new_acc)
print("Selected header:", first_header)

if new_acc != "P0DMV8":
    print("WARNING: first record is not P0DMV8. Check manually before continuing.")

out_path = RAW_FASTA_DIR / f"{new_acc}.fasta"
out_path.write_text(first_record + "\n")

print("Saved:", out_path)

In [ ]:
rec = read_single_fasta(RAW_FASTA_DIR / "P0DMV8.fasta")

print("ok:", rec["ok"])
print("error:", rec["error"])
print("length:", rec["length"])
print("header:", rec["header"])
print("seq first 80:", rec["seq"][:80] if rec["ok"] else "")

۲. حالا P08107 را در model-ready به P0DMV8 تبدیل کن

In [ ]:
old_acc = "P08107"
new_acc = "P0DMV8"

pairs_mr = pd.read_csv(PAIR_DIR / "pairs_all_model_ready.csv", dtype=str, low_memory=False)
pairs_mr["label"] = pairs_mr["label"].astype(int)

# Backup
pairs_mr.to_csv(
    PAIR_DIR / "pairs_all_model_ready_before_P08107_fix.csv",
    index=False
)

affected = pairs_mr[
    (pairs_mr["enz_ac"].astype(str) == old_acc) |
    (pairs_mr["sub_ac"].astype(str) == old_acc)
].copy()

print("Affected rows:", affected.shape)
display(affected[[
    "pair_id", "enzyme_class", "enz_ac", "sub_ac",
    "enz_gene", "sub_gene", "label", "source", "pmid"
]])

pairs_mr["enz_ac"] = pairs_mr["enz_ac"].replace(old_acc, new_acc)
pairs_mr["sub_ac"] = pairs_mr["sub_ac"].replace(old_acc, new_acc)

pairs_mr["pair_id"] = (
    pairs_mr["enzyme_class"].astype(str)
    + "|"
    + pairs_mr["enz_ac"].astype(str)
    + "|"
    + pairs_mr["sub_ac"].astype(str)
)

pairs_mr["group_id"] = (
    pairs_mr["enzyme_class"].astype(str)
    + "|"
    + pairs_mr["enz_ac"].astype(str)
)

dup = pairs_mr[pairs_mr.duplicated("pair_id", keep=False)].sort_values("pair_id")

conflict_keys = (
    pairs_mr.groupby("pair_id")["label"]
    .nunique()
    .reset_index(name="n_labels")
)

conflict_keys = conflict_keys[conflict_keys["n_labels"] > 1]

conflicts = pairs_mr[
    pairs_mr["pair_id"].isin(conflict_keys["pair_id"])
].sort_values("pair_id")

print("After replacement")
print("shape:", pairs_mr.shape)
print("unique pair_id:", pairs_mr["pair_id"].nunique())
print("duplicate rows:", len(dup))
print("label conflict pair_ids:", len(conflict_keys))

display(dup.head(50))
display(conflicts.head(50))

In [ ]:
if len(dup) == 0 and len(conflict_keys) == 0:
    pairs_mr.to_csv(
        PAIR_DIR / "pairs_all_model_ready.csv",
        index=False
    )

    pairs_mr[pairs_mr["label"] == 1].to_csv(
        PAIR_DIR / "pairs_positive_model_ready.csv",
        index=False
    )

    pairs_mr[pairs_mr["label"] == 0].to_csv(
        PAIR_DIR / "pairs_negative_model_ready.csv",
        index=False
    )

    required_enzymes_model = (
        pairs_mr[["enz_ac"]]
        .dropna()
        .drop_duplicates()
        .sort_values("enz_ac")
        .reset_index(drop=True)
    )

    required_substrates_model = (
        pairs_mr[["sub_ac"]]
        .dropna()
        .drop_duplicates()
        .sort_values("sub_ac")
        .reset_index(drop=True)
    )

    required_accessions_model = pd.DataFrame({
        "accession": sorted(
            set(required_enzymes_model["enz_ac"].astype(str)) |
            set(required_substrates_model["sub_ac"].astype(str))
        )
    })

    required_enzymes_model.to_csv(
        PAIR_DIR / "required_enzymes_model_ready.csv",
        index=False
    )

    required_substrates_model.to_csv(
        PAIR_DIR / "required_substrates_model_ready.csv",
        index=False
    )

    required_accessions_model.to_csv(
        PAIR_DIR / "required_accessions_model_ready.csv",
        index=False
    )

    pd.DataFrame([{
        "old_accession": old_acc,
        "new_accession": new_acc,
        "gene": "HSPA1A",
        "n_affected_pairs": len(affected),
        "reason": "P08107 returned empty FASTA; replaced with UniProt primary HSPA1A entry from gene search.",
    }]).to_csv(
        QC_DIR / "P08107_to_P0DMV8_replacement_log.csv",
        index=False
    )

    print("Saved replacement.")
    print("required_accessions_model:", required_accessions_model.shape)

else:
    print("Do not save. Duplicate or label conflict detected.")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

PROJECT_ROOT = Path(".")

PAIR_DIR = PROJECT_ROOT / "Data_proc" / "pairs"
RAW_FASTA_DIR = PROJECT_ROOT / "Data_raw" / "uniprot" / "uniprot_fasta"
INTERIM_UNIPROT_DIR = PROJECT_ROOT / "Data_interim" / "uniprot"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

for d in [INTERIM_UNIPROT_DIR, QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PAIRS_ALL_PATH = PAIR_DIR / "pairs_all_final.csv"
REQ_ACCESSIONS_PATH = PAIR_DIR / "required_accessions_model_ready.csv"
REQ_ENZYMES_PATH = PAIR_DIR / "required_enzymes_model_ready.csv"
REQ_SUBSTRATES_PATH = PAIR_DIR / "required_substrates_model_ready.csv"

print("pairs_all exists:", PAIRS_ALL_PATH.exists(), PAIRS_ALL_PATH)
print("required_accessions exists:", REQ_ACCESSIONS_PATH.exists(), REQ_ACCESSIONS_PATH)
print("required_enzymes exists:", REQ_ENZYMES_PATH.exists(), REQ_ENZYMES_PATH)
print("required_substrates exists:", REQ_SUBSTRATES_PATH.exists(), REQ_SUBSTRATES_PATH)
print("raw fasta dir exists:", RAW_FASTA_DIR.exists(), RAW_FASTA_DIR)

pairs_all = pd.read_csv(PAIRS_ALL_PATH, dtype=str, low_memory=False)
required_accessions = pd.read_csv(REQ_ACCESSIONS_PATH, dtype=str, low_memory=False)
required_enzymes = pd.read_csv(REQ_ENZYMES_PATH, dtype=str, low_memory=False)
required_substrates = pd.read_csv(REQ_SUBSTRATES_PATH, dtype=str, low_memory=False)

print("pairs_all:", pairs_all.shape)
print("required_accessions:", required_accessions.shape)
print("required_enzymes:", required_enzymes.shape)
print("required_substrates:", required_substrates.shape)

display(pairs_all.head())
display(required_accessions.head())

In [ ]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWYBXZUOJ*-")

def normalize_ac(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:
        return np.nan
    x = re.split(r"[;,|\s]", x)[0].strip()
    x = re.sub(r"-\d+$", "", x)
    return x.upper()


def extract_ac_from_header(header: str):
    """
    UniProt FASTA header usually:
    >sp|P12345|NAME_HUMAN ...
    """
    if not isinstance(header, str):
        return np.nan
    
    h = header.strip()
    if h.startswith(">"):
        h = h[1:].strip()
    
    if not h:
        return np.nan
    
    if "|" in h:
        parts = h.split("|")
        if len(parts) >= 2:
            return normalize_ac(parts[1])
    
    return normalize_ac(h.split()[0])


def read_single_fasta(path: Path):
    """
    Read one fasta file.
    Returns dict with status.
    """
    try:
        text = path.read_text(errors="ignore")
    except Exception as e:
        return {
            "ok": False,
            "error": f"read_error: {e}",
            "header": "",
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    if not text.strip():
        return {
            "ok": False,
            "error": "empty_file",
            "header": "",
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    
    if not lines:
        return {
            "ok": False,
            "error": "no_nonempty_lines",
            "header": "",
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    if not lines[0].startswith(">"):
        return {
            "ok": False,
            "error": "missing_header",
            "header": lines[0][:100],
            "seq": "",
            "header_ac": np.nan,
            "length": 0,
        }
    
    header = lines[0]
    seq = "".join(lines[1:]).replace(" ", "").replace("\t", "").upper()
    
    if not seq:
        return {
            "ok": False,
            "error": "empty_sequence",
            "header": header,
            "seq": "",
            "header_ac": extract_ac_from_header(header),
            "length": 0,
        }
    
    invalid_chars = sorted(set(seq) - VALID_AA)
    
    if invalid_chars:
        return {
            "ok": False,
            "error": "invalid_amino_acids:" + "".join(invalid_chars),
            "header": header,
            "seq": seq,
            "header_ac": extract_ac_from_header(header),
            "length": len(seq),
        }
    
    return {
        "ok": True,
        "error": "",
        "header": header,
        "seq": seq,
        "header_ac": extract_ac_from_header(header),
        "length": len(seq),
    }

In [ ]:
# Detect all single FASTA files
fasta_files = sorted(
    list(RAW_FASTA_DIR.glob("*.fasta")) +
    list(RAW_FASTA_DIR.glob("*.fa")) +
    list(RAW_FASTA_DIR.glob("*.faa"))
)

print("FASTA files found:", len(fasta_files))

index_rows = []

for i, path in enumerate(fasta_files):
    if i % 1000 == 0:
        print("processing", i, "/", len(fasta_files))
    
    file_stem_ac = normalize_ac(path.stem)
    rec = read_single_fasta(path)
    
    index_rows.append({
        "file_path": str(path),
        "file_name": path.name,
        "file_stem_ac": file_stem_ac,
        "header_ac": rec["header_ac"],
        "ok": rec["ok"],
        "error": rec["error"],
        "length": rec["length"],
        "header": rec["header"],
    })

fasta_index = pd.DataFrame(index_rows)

print("FASTA index:", fasta_index.shape)
print("OK FASTA:", int(fasta_index["ok"].sum()) if len(fasta_index) else 0)
print("Bad FASTA:", int((~fasta_index["ok"]).sum()) if len(fasta_index) else 0)

display(fasta_index.head())
display(fasta_index["error"].value_counts(dropna=False).head(20))

In [ ]:
bad_acc = "P08107"

bad_pairs = model_ready[
    (model_ready["enz_ac"].astype(str) == bad_acc) |
    (model_ready["sub_ac"].astype(str) == bad_acc)
].copy()

print("Rows using P08107:", bad_pairs.shape)

display(bad_pairs[[
    "pair_id", "enzyme_class", "enz_ac", "sub_ac",
    "enz_gene", "sub_gene",
    "enzyme_type", "label", "source", "pmid"
]])

In [ ]:
# Normalize required accessions
if "accession" in required_accessions.columns:
    req_col = "accession"
elif "uniprot_ac" in required_accessions.columns:
    req_col = "uniprot_ac"
else:
    req_col = required_accessions.columns[0]

required_accessions["accession"] = required_accessions[req_col].map(normalize_ac)
required_accessions = (
    required_accessions[["accession"]]
    .dropna()
    .drop_duplicates()
    .sort_values("accession")
    .reset_index(drop=True)
)

# Prefer file_stem_ac matching; header_ac is secondary evidence
ok_index = fasta_index[fasta_index["ok"]].copy()

# If duplicate FASTA files for same AC exist, keep first
ok_by_stem = (
    ok_index
    .dropna(subset=["file_stem_ac"])
    .drop_duplicates(subset=["file_stem_ac"], keep="first")
    .set_index("file_stem_ac")
)

coverage_rows = []

for acc in required_accessions["accession"]:
    if acc in ok_by_stem.index:
        row = ok_by_stem.loc[acc]
        coverage_rows.append({
            "accession": acc,
            "has_fasta": True,
            "fasta_ok": True,
            "file_path": row["file_path"],
            "file_name": row["file_name"],
            "length": int(row["length"]),
            "header_ac": row["header_ac"],
            "error": "",
        })
    else:
        # Check if present but bad
        bad_match = fasta_index[fasta_index["file_stem_ac"] == acc]
        if len(bad_match):
            err = ";".join(bad_match["error"].astype(str).unique())
            coverage_rows.append({
                "accession": acc,
                "has_fasta": True,
                "fasta_ok": False,
                "file_path": bad_match.iloc[0]["file_path"],
                "file_name": bad_match.iloc[0]["file_name"],
                "length": int(bad_match.iloc[0]["length"]),
                "header_ac": bad_match.iloc[0]["header_ac"],
                "error": err,
            })
        else:
            coverage_rows.append({
                "accession": acc,
                "has_fasta": False,
                "fasta_ok": False,
                "file_path": "",
                "file_name": "",
                "length": 0,
                "header_ac": "",
                "error": "missing_file",
            })

fasta_coverage = pd.DataFrame(coverage_rows)

missing_required = fasta_coverage[~fasta_coverage["has_fasta"]].copy()
bad_required = fasta_coverage[
    fasta_coverage["has_fasta"] & (~fasta_coverage["fasta_ok"])
].copy()

print("Required accessions:", len(fasta_coverage))
print("Covered OK:", int(fasta_coverage["fasta_ok"].sum()))
print("Missing:", len(missing_required))
print("Bad:", len(bad_required))

display(fasta_coverage.head())
display(missing_required.head())
display(bad_required.head())

In [ ]:
def prepare_role_df(df, col_guess):
    out = df.copy()
    if col_guess in out.columns:
        col = col_guess
    elif "accession" in out.columns:
        col = "accession"
    elif "uniprot_ac" in out.columns:
        col = "uniprot_ac"
    else:
        col = out.columns[0]
    
    out["accession"] = out[col].map(normalize_ac)
    out = out[["accession"]].dropna().drop_duplicates()
    return out


enz_req = prepare_role_df(required_enzymes, "enz_ac")
sub_req = prepare_role_df(required_substrates, "sub_ac")

enz_cov = enz_req.merge(fasta_coverage, on="accession", how="left")
sub_cov = sub_req.merge(fasta_coverage, on="accession", how="left")

fasta_coverage_qc = pd.DataFrame([
    {
        "dataset": "all_required_accessions",
        "n_required": len(fasta_coverage),
        "n_fasta_ok": int(fasta_coverage["fasta_ok"].sum()),
        "n_missing": int((~fasta_coverage["has_fasta"]).sum()),
        "n_bad": int((fasta_coverage["has_fasta"] & ~fasta_coverage["fasta_ok"]).sum()),
        "coverage_fraction": float(fasta_coverage["fasta_ok"].mean()) if len(fasta_coverage) else np.nan,
        "mean_length_ok": float(fasta_coverage.loc[fasta_coverage["fasta_ok"], "length"].mean()) if fasta_coverage["fasta_ok"].any() else np.nan,
        "median_length_ok": float(fasta_coverage.loc[fasta_coverage["fasta_ok"], "length"].median()) if fasta_coverage["fasta_ok"].any() else np.nan,
    },
    {
        "dataset": "required_enzymes",
        "n_required": len(enz_cov),
        "n_fasta_ok": int(enz_cov["fasta_ok"].fillna(False).sum()),
        "n_missing": int((enz_cov["has_fasta"].fillna(False) == False).sum()),
        "n_bad": int((enz_cov["has_fasta"].fillna(False) & (enz_cov["fasta_ok"].fillna(False) == False)).sum()),
        "coverage_fraction": float(enz_cov["fasta_ok"].fillna(False).mean()) if len(enz_cov) else np.nan,
        "mean_length_ok": float(enz_cov.loc[enz_cov["fasta_ok"] == True, "length"].mean()) if (enz_cov["fasta_ok"] == True).any() else np.nan,
        "median_length_ok": float(enz_cov.loc[enz_cov["fasta_ok"] == True, "length"].median()) if (enz_cov["fasta_ok"] == True).any() else np.nan,
    },
    {
        "dataset": "required_substrates",
        "n_required": len(sub_cov),
        "n_fasta_ok": int(sub_cov["fasta_ok"].fillna(False).sum()),
        "n_missing": int((sub_cov["has_fasta"].fillna(False) == False).sum()),
        "n_bad": int((sub_cov["has_fasta"].fillna(False) & (sub_cov["fasta_ok"].fillna(False) == False)).sum()),
        "coverage_fraction": float(sub_cov["fasta_ok"].fillna(False).mean()) if len(sub_cov) else np.nan,
        "mean_length_ok": float(sub_cov.loc[sub_cov["fasta_ok"] == True, "length"].mean()) if (sub_cov["fasta_ok"] == True).any() else np.nan,
        "median_length_ok": float(sub_cov.loc[sub_cov["fasta_ok"] == True, "length"].median()) if (sub_cov["fasta_ok"] == True).any() else np.nan,
    },
])

display(fasta_coverage_qc)

مرحله بعد: ساخت FASTAهای نهایی

In [ ]:
# Create accession -> fasta record map
def make_fasta_record(acc, row):
    rec = read_single_fasta(Path(row["file_path"]))
    if not rec["ok"]:
        return None
    
    seq = rec["seq"]
    return f">{acc}\n{seq}\n"


ok_cov = fasta_coverage[fasta_coverage["fasta_ok"]].copy()
ok_cov_by_acc = ok_cov.drop_duplicates("accession").set_index("accession")


def write_multifasta(accessions, out_path: Path):
    written = 0
    skipped = []
    
    with open(out_path, "w") as out:
        for acc in accessions:
            acc = normalize_ac(acc)
            
            if pd.isna(acc) or acc not in ok_cov_by_acc.index:
                skipped.append(acc)
                continue
            
            rec = make_fasta_record(acc, ok_cov_by_acc.loc[acc])
            
            if rec is None:
                skipped.append(acc)
                continue
            
            out.write(rec)
            written += 1
    
    print(out_path.name, "written:", written, "skipped:", len(skipped))
    return skipped


enz_skipped = write_multifasta(
    enz_req["accession"].tolist(),
    INTERIM_UNIPROT_DIR / "enzymes_model_ready.fasta"
)

sub_skipped = write_multifasta(
    sub_req["accession"].tolist(),
    INTERIM_UNIPROT_DIR / "substrates_model_ready.fasta"
)

all_skipped = write_multifasta(
    required_accessions["accession"].tolist(),
    INTERIM_UNIPROT_DIR / "required_sequences_model_ready.fasta"
)

pd.Series(enz_skipped, name="accession").to_csv(
    QC_DIR / "enzymes_model_ready_multifasta_skipped.csv",
    index=False
)

pd.Series(sub_skipped, name="accession").to_csv(
    QC_DIR / "substrates_model_ready_multifasta_skipped.csv",
    index=False
)

pd.Series(all_skipped, name="accession").to_csv(
    QC_DIR / "required_model_ready_multifasta_skipped.csv",
    index=False
)

بعد این ذخیره‌سازی و چک نهایی را اجرا کن:

In [ ]:
fasta_index.to_csv(
    INTERIM_UNIPROT_DIR / "fasta_index_all_single_files.csv",
    index=False
)

fasta_coverage.to_csv(
    INTERIM_UNIPROT_DIR / "fasta_index_model_ready.csv",
    index=False
)

missing_required.to_csv(
    QC_DIR / "missing_required_fastas_model_ready.csv",
    index=False
)

bad_required.to_csv(
    QC_DIR / "bad_required_fastas_model_ready.csv",
    index=False
)

fasta_coverage_qc.to_csv(
    QC_DIR / "fasta_coverage_qc_model_ready.csv",
    index=False
)

enz_cov.to_csv(
    QC_DIR / "fasta_coverage_required_enzymes_model_ready.csv",
    index=False
)

sub_cov.to_csv(
    QC_DIR / "fasta_coverage_required_substrates_model_ready.csv",
    index=False
)

print("Saved:")
print(INTERIM_UNIPROT_DIR / "fasta_index_model_ready.csv")
print(INTERIM_UNIPROT_DIR / "enzymes_model_ready.fasta")
print(INTERIM_UNIPROT_DIR / "substrates_model_ready.fasta")
print(INTERIM_UNIPROT_DIR / "required_sequences_model_ready.fasta")
print(QC_DIR / "fasta_coverage_qc_model_ready.csv")

و چک فایل‌ها:

In [ ]:
check_files = [
    INTERIM_UNIPROT_DIR / "fasta_index_all_single_files.csv",
    INTERIM_UNIPROT_DIR / "fasta_index_model_ready.csv",
    INTERIM_UNIPROT_DIR / "enzymes_model_ready.fasta",
    INTERIM_UNIPROT_DIR / "substrates_model_ready.fasta",
    INTERIM_UNIPROT_DIR / "required_sequences_model_ready.fasta",
    QC_DIR / "fasta_coverage_qc_model_ready.csv",
    QC_DIR / "missing_required_fastas_model_ready.csv",
    QC_DIR / "bad_required_fastas_model_ready.csv",
    QC_DIR / "fasta_coverage_required_enzymes_model_ready.csv",
    QC_DIR / "fasta_coverage_required_substrates_model_ready.csv",
    QC_DIR / "P08107_to_P0DMV8_replacement_log.csv",
    QC_DIR / "excluded_compound_accession_pairs.csv",
]

for f in check_files:
    print(f.name, "exists:", f.exists())
    
    if f.exists() and f.suffix == ".csv":
        tmp = pd.read_csv(f, dtype=str, low_memory=False)
        print("shape:", tmp.shape)
    
    elif f.exists() and f.suffix in {".fasta", ".fa", ".faa"}:
        n_headers = sum(1 for line in open(f, errors="ignore") if line.startswith(">"))
        print("records:", n_headers)